# Single Image to 3D — Gradio on Colab Free
Run the single code cell below. It checks GPU/HF_TOKEN, mounts Drive for resumable session outputs, installs pinned dependencies and creates a Gradio share link. `QUICK_TEST=True` keeps the Colab Free geometry-only preset; production generation still uses real pretrained backends only.

In [ ]:
import importlib.util, os, subprocess, sys, time
from pathlib import Path
STARTED = time.perf_counter()
QUICK_TEST = True
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, cwd / 'image-to-3d-shapenet', *cwd.parents] if (p / 'configs' / 'inference.yaml').is_file()), None)
if PROJECT_ROOT is None:
    clone_target = Path('/content/image-to-3d-shapenet')
    try:
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/dinhcongtiendat2003/image-to-3d-shapenet.git', str(clone_target)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Không tìm thấy hoặc clone được repository từ GitHub.') from exc
    PROJECT_ROOT = clone_target
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError('Cài dependency thất bại; Restart runtime rồi Run all lại.') from exc
import torch
CUDA = torch.cuda.is_available()
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3 if CUDA else 0.0
HF_TOKEN_AVAILABLE = bool(os.getenv('HF_TOKEN'))
try:
    from google.colab import userdata
    HF_TOKEN_AVAILABLE = HF_TOKEN_AVAILABLE or bool(userdata.get('HF_TOKEN'))
except Exception:
    pass
DRIVE_ROOT = None
try:
    from google.colab import drive
    mount = os.getenv('GOOGLE_DRIVE_MOUNT', '/content/drive')
    drive.mount(mount)
    DRIVE_ROOT = Path(mount) / 'MyDrive' / 'image-to-3d-shapenet'
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
except Exception as exc:
    DRIVE_ROOT = None
    print(f'Google Drive mount không khả dụng ({type(exc).__name__}); session sẽ lưu local trong /content.')
print({'quick_test': QUICK_TEST, 'cuda': CUDA, 'gpu': torch.cuda.get_device_name(0) if CUDA else None, 'vram_gb': round(VRAM_GB, 2), 'hf_token_available': HF_TOKEN_AVAILABLE, 'drive': str(DRIVE_ROOT) if DRIVE_ROOT else None})
BACKEND_ROOT = PROJECT_ROOT / '.cache' / 'backends'
TRIPOSR_ROOT = BACKEND_ROOT / 'TripoSR'
if CUDA and importlib.util.find_spec('tsr') is None:
    BACKEND_ROOT.mkdir(parents=True, exist_ok=True)
    if not TRIPOSR_ROOT.is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/VAST-AI-Research/TripoSR.git', str(TRIPOSR_ROOT)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'omegaconf==2.3.0', 'einops==0.7.0', 'imageio[ffmpeg]', 'xatlas==0.0.9', 'moderngl==5.10.0', 'git+https://github.com/tatsy/torchmcubes.git'], check=True)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if TRIPOSR_ROOT.is_dir():
    sys.path.insert(0, str(TRIPOSR_ROOT))
from image_to_3d_shapenet import load_config
from image_to_3d_shapenet.app.pipeline import AppInferenceService
from image_to_3d_shapenet.app.gradio_app import build_app, create_service
CONFIG = PROJECT_ROOT / 'configs' / 'inference.yaml'
config = load_config(CONFIG)
service = AppInferenceService(config, DRIVE_ROOT / 'gradio_sessions') if DRIVE_ROOT else create_service(CONFIG)
demo = build_app(CONFIG, service=service)
print({'session_output': str(service.output_root), 'resume': 'Mỗi session UUID được giữ nguyên; Run all lại không xóa phiên cũ.', 'setup_seconds': round(time.perf_counter() - STARTED, 2)})
try:
    demo.launch(server_name='0.0.0.0', share=True, allowed_paths=[str(service.output_root)], debug=True)
except Exception as exc:
    raise RuntimeError('Không tạo được Gradio share link. File session vẫn nằm trong session_output; thử launch share=False.') from exc
